# Script 1: ROI Analysis

## Imports and Set-Up

In [ ]:
import scipy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.markers import MarkerStyle
fillstyles=["right","left","bottom","top","right","left","bottom","top","right","left","bottom","top",]
import mne 
import os
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import gaussian
import mne_bids
import tempfile
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import ward, fcluster
from scipy.spatial import ConvexHull
from scipy import interpolate

In [ ]:
cwd = os.getcwd() 
print("working directory:",cwd)
bids_root = os.path.join(cwd, 'Data/BIDS')    
task = 'oddball'
bids_path = mne_bids.BIDSPath(subject="Go",session=str(1), task=task,
                            acquisition='01', run='01',
                            root=bids_root)

colors=['r','g','b','y','c','m']
Electrodes = []  
cwd = os.getcwd() + "/Data/"
for session,folder in enumerate(os.listdir(cwd)):

    mat_files = [f for f in os.listdir(os.path.join(cwd, folder)) if f.endswith('.mat') and 'Electrodes' in f]
    if mat_files:
        mat_path = os.path.join(cwd, folder, mat_files[0])
        Electrodes.append(sc.io.loadmat(mat_path))


Electrodes_Fr,Electrodes_Go,Electrodes_Kr = Electrodes[2]["LINE"],Electrodes[4]["LINE"],Electrodes[10] ["LINE"]
Lines=[Electrodes_Fr,Electrodes_Go,Electrodes_Kr]

## Functions

In [ ]:
def get_gfp_threshold(epochs, percentile=95):

    data = epochs.get_data(copy=False)
    
    gfp_per_epoch = np.std(data, axis=1)
    max_gfp_per_epoch = gfp_per_epoch.max(axis=1)
    
    
    # Determine the threshold
    thresh = np.percentile(max_gfp_per_epoch, percentile)
    return thresh

def standard_and_deviants(subject, subjects=["Fr", "Go", "Kr"],picks="all",highpass=0.3,lowpass=40,car=True):
    # Constants
    RESAMPLE_RATE = 1000
    NOTCH_FREQS = np.arange(50, RESAMPLE_RATE/2, 50)
    STANDARD_EVENT_OFFSET = 503
    EPOCH_TMIN, EPOCH_TMAX = -0.1, 0.25
    LARGE_TMIN, LARGE_TMAX = -0.603, 0.25

    all_epochs_deviant = []
    all_epochs_standard = []
    all_large_epochs = []
    bad_indices = []
    all_raws = []
    
    subject_name = subjects[subject]
    cwd = os.getcwd()

    # Get subject name and session directories
    subject_name = subjects[subject]
    sub_dirs = os.listdir(os.path.join(cwd, bids_root, f"sub-{subject_name}"))
    #print(bids_root)
    print(f"Processing {len(sub_dirs)} sessions for subject {subject_name}")

    for session_idx in range(len(sub_dirs)):
        session_num = session_idx + 1

        # Load raw data
        bids_path = mne_bids.BIDSPath(
            subject=subject_name,
            session=str(session_num),
            task=task,
            acquisition="01",
            run="01",
            root=bids_root,
        )
        # 1. Load Data
        raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose="CRITICAL")
        raw.load_data(verbose="CRITICAL")
        raw.apply_function(lambda x: x * 1e-6, verbose="CRITICAL")  # Convert to Volts

        # 2. Notch and High-Pass Filter
        raw.filter(highpass, lowpass, verbose="CRITICAL")
        raw.notch_filter(freqs=NOTCH_FREQS, verbose="CRITICAL")

        # 3. Identify Bad Channels BEFORE CAR
        raw.pick(picks, verbose="CRITICAL")
        raw_data = raw.get_data(verbose="CRITICAL")
        chan_std = np.std(raw_data, axis=1)
        z_scores = (chan_std - np.mean(chan_std)) / np.std(chan_std)
        bad_indices_idx = np.where(np.abs(z_scores) > 5)[0]
        #print(f"Session {session_idx}: Found bad channels: {bad_indices_idx}")

        bad_indices.append(bad_indices_idx)
        all_raws.append(raw)
    bad_indices = np.unique(np.concatenate(bad_indices))
    for raw in all_raws:
        if raw.info["sfreq"] != RESAMPLE_RATE:
            raw.resample(RESAMPLE_RATE, verbose="CRITICAL")
                # Mark bads in the info object
        raw.info['bads'] = [raw.ch_names[i] for i in bad_indices]
              # 5. Resample
    for session_idx in range(len(sub_dirs)):
        # 4. Apply Common Average Reference (CAR)
        raw = all_raws[session_idx]
        if car ==True:
            raw.set_eeg_reference(ref_channels="average", projection=False, verbose="CRITICAL")
  
        # 6. Extract Events & Epoching
        events, _ = mne.events_from_annotations(raw, verbose="CRITICAL")
        
        # Create standard events by shifting the timing
        standard_events = events.copy()
        standard_events[:, 0] -= int(STANDARD_EVENT_OFFSET * (RESAMPLE_RATE / 1000))
        

        # Define common kwargs for epoching
        epoch_params = dict(tmin=EPOCH_TMIN, tmax=EPOCH_TMAX, baseline=(-0.1, 0), 
                            preload=True, picks="ecog", verbose="CRITICAL")

        all_epochs_deviant.append(mne.Epochs(raw, events=events, **epoch_params))
        all_epochs_standard.append(mne.Epochs(raw, events=standard_events, **epoch_params))
        
        # Large Epoch for sync rejection
        all_large_epochs.append(mne.Epochs(raw, events=events, 
                                           tmin=LARGE_TMIN, tmax=LARGE_TMAX, 
                                           baseline=(-0.1, 0), preload=True, picks="ecog"))

    # 7. Concatenate and Final Rejection
    epochs_deviant = mne.concatenate_epochs(all_epochs_deviant, verbose="CRITICAL")
    epochs_standard = mne.concatenate_epochs(all_epochs_standard, verbose="CRITICAL")
    large_epochs = mne.concatenate_epochs(all_large_epochs, verbose="CRITICAL")

    # Use the GFP-based thresholding on the large epochs
    gfp_thresh = get_gfp_threshold(large_epochs, percentile=95)
    
    # Calculate GFP peak per epoch
    data = large_epochs.get_data(copy=False)
    max_gfps = np.std(data, axis=1).max(axis=1)
    bad_epoch_mask = max_gfps > gfp_thresh
    indices_to_drop = np.where(bad_epoch_mask)[0]

    # Drop from all sets to maintain synchronization
    large_epochs.drop(indices_to_drop, reason="GFP_THRESHOLD",verbose="CRITICAL")
    epochs_deviant.drop(indices_to_drop, reason="GFP_THRESHOLD", verbose="CRITICAL")
    epochs_standard.drop(indices_to_drop, reason="GFP_THRESHOLD", verbose="CRITICAL")

    return epochs_standard, epochs_deviant, large_epochs,all_raws

In [ ]:
def get_coords(epochs):
    ch_pos = epochs.get_montage()._get_ch_pos()
    coords = np.array(list(ch_pos.values()))
    x, y, _ = coords.T * 1000
    return x, y
def plot_electrode_map(ax, x, y, line_img=None,numbered=False, background=None):
    """Base electrode layout for the map panel."""
    if line_img is not None:
        if background is None:
            outline=np.argwhere((np.average(line_img,axis=2)<220))
            ax.scatter(outline[:,1],outline[:,0],c='k',s=0.03)
            ax.set_aspect('equal', adjustable='box')
            ax.invert_yaxis()
        else:
            ax.imshow(line_img)    
    ax.scatter(x, y, c='k', alpha=0.2, s=10, label='All Electrodes')
    if numbered:
        for i, (x_i, y_i) in enumerate(zip(x, y)):
            ax.text(x_i + 10, y_i + 10, str(i), fontsize=6) 
    ax.axis('off')

def plot_erp_panel(ax, times_ms, avg_stand, avg_dev, title, ylabel='Amplitude (V)', invert_y=False):
    """Standard ERP waveform panel."""
    ax.plot(times_ms, avg_stand, label='Standard', color='blue', lw=1.5)
    ax.plot(times_ms, avg_dev,   label='Deviant',  color='green', lw=1.5)
    ax.plot(times_ms, avg_dev - avg_stand, label='Diff', color='red', linestyle=':', alpha=0.8)
    ax.axhline(0, color='k', linestyle='--', alpha=0.5)
    ax.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax.set(title=title, ylabel=ylabel, xlabel='Time (ms)')
    if invert_y:
        ax.yaxis.set_inverted(True)
    ax.legend(loc='upper left', fontsize='small')

In [ ]:
def ward_clustering_on_epochs(epochs):
    if type(epochs) == mne.Epochs:
        epochs = epochs.get_data() 

    n_epochs, n_channels, n_times = epochs.shape

    dist_sum = np.zeros((n_channels * (n_channels - 1)) // 2)

    for i in range(n_epochs):
        # Correlation distance for this specific epoch
        d = pdist(epochs[i, :, :], metric='correlation')
        dist_sum += d

    avg_dist = dist_sum / n_epochs

    Z = ward(avg_dist)
    cluster = fcluster(Z, t=0.5 * Z[:, 2].max(), criterion='distance')

    print(f"Found {len(np.unique(cluster))} clusters across {n_channels} electrodes.")
    return cluster

def cluster_testing(data_stand, data_dev,sig_threshold=0.05,subject=0):
    name=["Fr", "Go", "Kr"][subject]
    Line=Lines[subject]
    adj = mne.channels.find_ch_adjacency(
        data_stand.set_channel_types(dict(zip(data_stand.get_montage().ch_names, ["eeg"] * len(data_stand.get_montage().ch_names)))).info,
        ch_type='eeg')[0]
    
    X_dev = np.transpose(data_dev.get_data(tmin=0), [0, 2, 1])
    X_std = np.transpose(data_stand.get_data(tmin=0), [0, 2, 1])
    X_diff = X_dev - X_std
    
    p_threshold = 0.05
    df = X_diff.shape[0] - 1
    threshold = sc.stats.distributions.t.ppf(1 - p_threshold / 2, df=df)
    
    T_obs, clusters, cluster_p_values, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_diff, adjacency=adj, n_jobs=-1, max_step=5, threshold=threshold, tail=0
    )
    
    for i_c, c in enumerate(clusters):
        if cluster_p_values[i_c] <= p_threshold:
            print(i_c)
    sig_mask = cluster_p_values <= sig_threshold
    sig_clusters = np.array(clusters, dtype=object)[sig_mask]
    
    electrodes_in_clusters=[]
    for cluster in sig_clusters:
        electrodes_in_clusters.extend(list(set(cluster[1])))
    plot_clusters(data_stand, data_dev, clusters, cluster_p_values, electrodes_in_clusters, subject=name,Line=Line)
    return T_obs,clusters, cluster_p_values, electrodes_in_clusters

def plot_clusters(data_stand, data_dev, clusters, cluster_p_values, electrodes_in_clusters, subject="Unknown", Line=None,sig_threshold=0.05):
    sig_mask = cluster_p_values <= sig_threshold
    sig_clusters = np.array(clusters, dtype=object)[sig_mask]
    n_occurence=np.unique(electrodes_in_clusters,return_counts=True)
    n_occurence=dict(zip(n_occurence[0],n_occurence[1]))


    sig_pvals = np.array(cluster_p_values)[sig_mask]
    n_sig = len(sig_clusters)
    
    fig, ax = plt.subplots(1, n_sig + 1, figsize=(5 + 5 * n_sig, 4), squeeze=False)
    ax = ax[0]

    x, y = get_coords(data_stand)

    plot_electrode_map(ax[0], x, y, line_img=Line)
    
    # Define colors for clusters using tab10 colormap
    colors = plt.get_cmap( "plasma",n_sig + 1).colors
  
    
    for i_c, (cluster, p_val) in enumerate(zip(sig_clusters, sig_pvals)):
        time_idx = cluster[0].astype(int)+100
        chan_idx = list(set(cluster[1].astype(int)))
        label_added = False
        for chan in chan_idx:
            curr_label = f'Cluster {i_c}' if not label_added else None
            if n_occurence[chan]==1:
                # 2. Matching Map Scatter Style
                ax[0].scatter(x[chan], y[chan], color=colors[i_c], label=curr_label, s=40)

            else:
                ax[0].scatter(x[chan], y[chan], color=colors[i_c], label=curr_label, s=40, marker=MarkerStyle("o", fillstyle=fillstyles[i_c]))
            label_added=True
            
        # 3. Waveform calculations
        avg_stand = np.mean(data_stand.get_data()[:, chan_idx], axis=(0, 1))
        avg_dev = np.mean(data_dev.get_data()[:, chan_idx], axis=(0, 1))
        full_times = data_stand.times * 1000 
        
        ax_i = ax[i_c + 1]
        t_start = data_stand.times[time_idx[0]] * 1000
        t_end = data_stand.times[time_idx[-1]] * 1000
        ax_i.axvspan(t_start, t_end, color='red', alpha=0.3)
        
        # 4. Matching Line Styling (lw, colors, alpha)
        plot_erp_panel(ax_i, full_times, avg_stand, avg_dev, title=f'Cluster {i_c} (p={p_val:.3f})', invert_y=False)

    
    # Final Legend and Title adjustment
    ax[0].legend(fontsize='small', loc='lower right')
    ax[0].get_xaxis().set_visible(False)
    ax[0].get_yaxis().set_visible(False)
    fig.suptitle(f"Monkey {subject} – Significant Cluster Analysis", fontsize=14)
    return fig, ax

def plot_clusters_mann_whitney_u(data_stand, data_dev, cluster, signif_idx_chx, subject, wilc_corrected=None, Line=None):
    # Setup the figure
    num_clusters = max(cluster)
    fig, ax = plt.subplots(1, num_clusters + 1, figsize=(5 + 5 * num_clusters, 4), squeeze=False,constrained_layout=True)
    ax = ax[0]
    
    x, y = get_coords(data_stand)
    
    plot_electrode_map(ax[0], x, y, line_img=Line)   
    # Define colors for clusters
    colors = plt.get_cmap( "plasma",num_clusters + 1).colors

    # Loop through each identified cluster
    for clx_idx, clx in enumerate(sorted(set(cluster))):
        # 1. Map the cluster electrodes
        chx_in_cluster_indices = [i for i, c in enumerate(cluster) if c == clx]
        chx_in_cluster_true = signif_idx_chx[np.array(chx_in_cluster_indices).astype(int)]
        
        ax[0].scatter(x[chx_in_cluster_true], y[chx_in_cluster_true], 
                      color=colors[clx_idx], label=f'Cluster {clx}', s=40)

        # 2. Extract Full ERP Waveforms (including baseline)
        avg_stand = np.mean(data_stand.get_data()[:, chx_in_cluster_true], axis=(0, 1))
        avg_dev = np.mean(data_dev.get_data()[:, chx_in_cluster_true], axis=(0, 1))
        full_times = data_stand.times * 1000 # Convert seconds to ms
        
        # 3. Create Significance Count Array
        sig_matrix = wilc_corrected[chx_in_cluster_true, :] < 0.05
        sig_count_post = sig_matrix.sum(axis=0)
        
        # Align post-stimulus sig_count with full_times
        sig_count_full = np.zeros(len(full_times))
        # Find index where t=0 starts
        zero_idx = np.argmin(np.abs(full_times - 0)) 
        # Fill from zero_idx to the end (ensuring we don't exceed array bounds)
        end_idx = min(zero_idx + len(sig_count_post), len(sig_count_full))
        sig_count_full[zero_idx:end_idx] = sig_count_post[:end_idx-zero_idx]
        sig_count_full=(sig_count_full/len(chx_in_cluster_true))*100

        # 4. Plot ERPs on ax[clx]
        plot_erp_panel(ax[clx], full_times, avg_stand, avg_dev, title=f'Cluster {clx} (n={len(chx_in_cluster_true)})', invert_y=False)

        # 5. Add Secondary Axis for Significance Count
        ax_twin = ax[clx].twinx()
        ax_twin.fill_between(full_times, 0, sig_count_full, color='gray', alpha=0.3, label='Sig. Electrodes (%)')
        ax_twin.set_ylim(0, 100)
        ax_twin.set_ylabel('Sig. Electrodes (%)', color='gray', fontsize=9)
        ax_twin.tick_params(axis='y', labelcolor='gray')

        # Formatting
        ax[clx].axhline(0, color='k', linestyle='--', alpha=0.5)
        ax[clx].axvline(0, color='k', linestyle='--', alpha=0.5)
        ax[clx].set_title(f'Cluster {clx} (n={len(chx_in_cluster_true)})')
        ax[clx].set_ylabel('Amplitude (uV)')
        ax[clx].set_xlabel('Time (ms)')

        ax[clx].legend(loc='upper left', fontsize='small')
    ax[0].get_xaxis().set_visible(False)
    ax[0].get_yaxis().set_visible(False)
    ax[0].legend(fontsize='small', loc='lower right')
    fig.suptitle(f"Monkey {subject} – Significant Cluster Analysis", fontsize=14)
    return fig, ax


def cluster_testing_mann_whitney_u(data_stand, data_dev,plotting=True,subject=0):
    # 1. Use Mann-Whitney U if 'Rank Sum' is intended
    stat, pvals = sc.stats.mannwhitneyu(data_dev.get_data(tmin=0), data_stand.get_data(tmin=0), axis=0)
    name=["Fr", "Go", "Kr"][subject]
    Line=Lines[subject]
    # 2. FDR Correction 
    p_flat = pvals.flatten()

    wilc_corrected = sc.stats.false_discovery_control(p_flat).reshape(pvals.shape)

    # 3. Identify Significant Channels 
    significant_mask = np.any(wilc_corrected < 0.05, axis=1)
    signif_idx_chx = np.where(significant_mask)[0]
    if len(signif_idx_chx) == 0:
        print("No significant channels found after FDR correction.")
        return None, None, None

    # 4. Clustering on significant difference waves
    data_dev_crop = data_dev.copy().crop(tmin=0)
    data_stand_crop = data_stand.copy().crop(tmin=0)
    diff_waves = (data_dev_crop.average(picks=signif_idx_chx).get_data() - 
                data_stand_crop.average(picks=signif_idx_chx).get_data())
    diff_waves = data_dev_crop.pick(picks=signif_idx_chx).get_data() - data_stand_crop.pick(picks=signif_idx_chx).get_data()
    cluster=ward_clustering_on_epochs(diff_waves)
    if plotting:
        plot_clusters_mann_whitney_u(data_stand, data_dev, cluster, signif_idx_chx,name,wilc_corrected,Line) 
    return cluster, signif_idx_chx, wilc_corrected

## Load Data

In [ ]:
picks_indices=[[7,9,11,31,24,28,1,17],[32,52,63,56,47,40,31,7],[32,51,61,55,45,40,28,7]] #Electrodes for Fr, Go, Kr

In [ ]:
# Highpassed Gamma Data
ep_fr_stand_gamma,ep_fr_dev_gamma,_,_=standard_and_deviants(0,highpass=30,lowpass=None)
ep_go_stand_gamma,ep_go_dev_gamma,_,_=standard_and_deviants(1,highpass=30,lowpass=None)
ep_kr_stand_gamma,ep_kr_dev_gamma,_,_=standard_and_deviants(2,highpass=30,lowpass=None)

# Lowpassed Data
ep_fr_stand_lp,ep_fr_dev_lp,_,_=standard_and_deviants(0,lowpass=30)
ep_go_stand_lp,ep_go_dev_lp,_,_=standard_and_deviants(1,lowpass=30)
ep_kr_stand_lp,ep_kr_dev_lp,_,_=standard_and_deviants(2,lowpass=30)

# filtered data like resampled at 500 Hz
ep_fr_stand,ep_fr_dev,_,all_fr_raws=standard_and_deviants(0,lowpass=250)
ep_go_stand,ep_go_dev,_,all_go_raws=standard_and_deviants(1,lowpass=250)
ep_kr_stand,ep_kr_dev,_,all_kr_raws=standard_and_deviants(2,lowpass=250)


## Permutation Cluster Test

In [ ]:
T_obs_fr_lp,clusters_fr_lp, cluster_p_values_fr_lp, electrodes_in_clusters_fr_lp= cluster_testing(data_stand=ep_fr_stand_lp, data_dev=ep_fr_dev_lp, subject=0)
T_obs_go_lp,clusters_go_lp, cluster_p_values_go_lp, electrodes_in_clusters_go_lp = cluster_testing(data_stand=ep_go_stand_lp, data_dev=ep_go_dev_lp, subject=1)
T_obs_kr_lp,clusters_kr_lp, cluster_p_values_kr_lp, electrodes_in_clusters_kr_lp = cluster_testing(data_stand=ep_kr_stand_lp, data_dev=ep_kr_dev_lp, subject=2)

T_obs_fr_gamma,clusters_fr_gamma, cluster_p_values_fr_gamma,electrodes_in_clusters_fr_gamma = cluster_testing(data_stand=ep_fr_stand_gamma, data_dev=ep_fr_dev_gamma, subject=0)
T_obs_go_gamma,clusters_go_gamma, cluster_p_values_go_gamma,electrodes_in_clusters_go_gamma = cluster_testing(data_stand=ep_go_stand_gamma, data_dev=ep_go_dev_gamma, subject=1)
T_obs_kr_gamma,clusters_kr_gamma, cluster_p_values_kr_gamma,electrodes_in_clusters_kr_gamma = cluster_testing(data_stand=ep_kr_stand_gamma, data_dev=ep_kr_dev_gamma, subject=2)

In [ ]:
fig, ax =plt.subplots(1,3,figsize=(15,5))
data={"Fr": (clusters_fr_lp, cluster_p_values_fr_lp,clusters_fr_gamma, cluster_p_values_fr_gamma, ep_fr_dev_lp),
      "Go": (clusters_go_lp, cluster_p_values_go_lp,clusters_go_gamma, cluster_p_values_go_gamma, ep_go_dev_lp),
      "Kr": (clusters_kr_lp, cluster_p_values_kr_lp,clusters_kr_gamma, cluster_p_values_kr_gamma, ep_kr_dev_lp)}
for i, (subject, (clusters, cluster_p_values, clusters_gamma, cluster_p_values_gamma, ep_stand_lp)) in enumerate(data.items()):
    
    x,y = get_coords(ep_stand_lp)
    gamma_clusters = [np.unique(cluster[1]) for cluster, p_val in zip(clusters_gamma, cluster_p_values_gamma) if p_val < 0.05]
    if len(gamma_clusters) > 0:
        chx_idxs = np.unique(np.concatenate(gamma_clusters))
        print(f"Subject {subject} has significant gamma clusters in channels: {chx_idxs}")

        x_coords, y_coords = get_coords(ep_stand_lp.copy().pick(chx_idxs))
        points = np.column_stack((x_coords, y_coords))

        if len(points) >= 3:
            hull = ConvexHull(points, qhull_options="QJ")
            hull_points = points[hull.vertices]
            hull_points = np.append(hull_points, [hull_points[0]], axis=0)

            tck, u = interpolate.splprep([hull_points[:, 0], hull_points[:, 1]], s=0, per=True)
            u_new = np.linspace(0, 1, 100)
            x_smooth, y_smooth = interpolate.splev(u_new, tck)

            ax[i].plot(x_smooth, y_smooth, color='red', lw=0.8,linestyle="--", label='Gamma Cluster Boundary')
            ax[i].fill(x_smooth, y_smooth, color='red', alpha=0.1, hatch='//',facecolor='none')
        elif len(points) == 2:
            ax[i].plot(points[:, 0], points[:, 1], color='red', lw=0.8)
        elif len(points) == 1:
            ax[i].scatter(points[0, 0], points[0, 1], color='red', s=30)

        ax[i].set_aspect('equal')

    chx_idxs = [np.unique(cluster[1]) for cluster, p_val in zip(clusters, cluster_p_values) if p_val < 0.05]
    colors = plt.get_cmap("viridis", len(chx_idxs) + 1).colors

    for n, chx_idx in enumerate(chx_idxs):
        x_coords, y_coords = get_coords(ep_stand_lp.copy().pick(np.atleast_1d(chx_idx)))
        points = np.column_stack((x_coords, y_coords))

        if len(points) >= 3:
            hull = ConvexHull(points, qhull_options="QJ")
            hull_points = points[hull.vertices]
            hull_points = np.append(hull_points, [hull_points[0]], axis=0)

            tck, u = interpolate.splprep([hull_points[:, 0], hull_points[:, 1]], s=0, per=True)
            u_new = np.linspace(0, 1, 100)
            x_smooth, y_smooth = interpolate.splev(u_new, tck)

            ax[i].fill(x_smooth, y_smooth, color=colors[n], alpha=0.15, label=f'Cluster Area {n+1}')
            ax[i].plot(x_smooth, y_smooth, color=colors[n], lw=1, alpha=0.3)
        elif len(points) == 2:
            ax[i].plot(points[:, 0], points[:, 1], color=colors[n], lw=1, alpha=0.3)
        elif len(points) == 1:
            ax[i].scatter(points[0, 0], points[0, 1], color=colors[n], s=25)

    plot_electrode_map(ax[i], x, y, line_img=Lines[i])

fig.tight_layout()
plt.savefig("significant_perm_clusters_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

## Wilcoxon Rank Sum Test with FDR correction and cluster plotting

In [ ]:
cluster_fr_gamma, signif_idx_chx_fr_gamma, wilc_corrected_fr_gamma = cluster_testing_mann_whitney_u(ep_fr_stand_gamma, ep_fr_dev_gamma,subject=0)
cluster_go_gamma, signif_idx_chx_go_gamma, wilc_corrected_go_gamma = cluster_testing_mann_whitney_u(ep_go_stand_gamma, ep_go_dev_gamma,subject=1)
cluster_kr_gamma, signif_idx_chx_kr_gamma, wilc_corrected_kr_gamma = cluster_testing_mann_whitney_u(ep_kr_stand_gamma, ep_kr_dev_gamma,subject=2)

In [ ]:
cluster_fr_lp, signif_idx_chx_fr_lp, wilc_corrected_fr_lp = cluster_testing_mann_whitney_u(ep_fr_stand_lp, ep_fr_dev_lp,subject=0)
cluster_go_lp, signif_idx_chx_go_lp, wilc_corrected_go_lp = cluster_testing_mann_whitney_u(ep_go_stand_lp, ep_go_dev_lp,subject=1)
cluster_kr_lp, signif_idx_chx_kr_lp, wilc_corrected_kr_lp = cluster_testing_mann_whitney_u(ep_kr_stand_lp, ep_kr_dev_lp,subject=2)

In [ ]:
fig, ax =plt.subplots(1,3,figsize=(15,5))
cluster_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
                  "#8172B2", "#937860"] 
 
x,y = get_coords(ep_fr_dev_lp)
plot_electrode_map(ax[0], x, y, line_img=Lines[0])
x_lp,y_lp = get_coords(ep_fr_dev_lp.copy().pick(signif_idx_chx_fr_lp))
x_gamma,y_gamma = get_coords(ep_fr_dev_gamma.copy().pick(signif_idx_chx_fr_gamma))
cluster_fr_lp = ward_clustering_on_epochs(ep_fr_dev_lp.copy().pick(signif_idx_chx_fr_lp).get_data()-ep_fr_stand_lp.copy().pick(signif_idx_chx_fr_lp).get_data())
cluster_fr_gamma = ward_clustering_on_epochs(ep_fr_dev_gamma.copy().pick(signif_idx_chx_fr_gamma).get_data()-ep_fr_stand_gamma.copy().pick(signif_idx_chx_fr_gamma).get_data())

for clx_idx, clx in enumerate(sorted(set(cluster_fr_lp))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_fr_lp) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[0].scatter(x_lp[chx_in_cluster_true], y_lp[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} < 30 Hz', s=20)
for clx_idx, clx in enumerate(sorted(set(cluster_fr_gamma))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_fr_gamma) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[0].scatter(x_gamma[chx_in_cluster_true], y_gamma[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} > 30 Hz', s=70,facecolors='none', edgecolors=cluster_colors[clx_idx],alpha=1)

x,y = get_coords(ep_go_stand_lp)
plot_electrode_map(ax[1], x, y, line_img=Lines[1])
x_lp,y_lp = get_coords(ep_go_stand_lp.copy().pick(signif_idx_chx_go_lp))
x_gamma,y_gamma = get_coords(ep_go_stand_gamma.copy().pick(signif_idx_chx_go_gamma))

cluster_go_lp = ward_clustering_on_epochs(ep_go_stand_lp.copy().pick(signif_idx_chx_go_lp).get_data()-ep_go_dev_lp.copy().pick(signif_idx_chx_go_lp).get_data())
cluster_go_gamma = ward_clustering_on_epochs(ep_go_stand_gamma.copy().pick(signif_idx_chx_go_gamma).get_data()-ep_go_dev_gamma.copy().pick(signif_idx_chx_go_gamma).get_data())
for clx_idx, clx in enumerate(sorted(set(cluster_go_lp))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_go_lp) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[1].scatter(x_lp[chx_in_cluster_true], y_lp[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} < 30 Hz', s=20)
for clx_idx, clx in enumerate(sorted(set(cluster_go_gamma))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_go_gamma) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[1].scatter(x_gamma[chx_in_cluster_true], y_gamma[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} > 30 Hz', s=70,facecolors='none', edgecolors=cluster_colors[clx_idx],alpha=1)

x,y = get_coords(ep_kr_stand_lp)
plot_electrode_map(ax[2], x, y, line_img=Lines[2])
x_lp,y_lp = get_coords(ep_kr_stand_lp.copy().pick(signif_idx_chx_kr_lp))
x_gamma,y_gamma = get_coords(ep_kr_stand_gamma.copy().pick(signif_idx_chx_kr_gamma))
cluster_kr_lp = ward_clustering_on_epochs(ep_kr_stand_lp.copy().pick(signif_idx_chx_kr_lp).get_data()-ep_kr_dev_lp.copy().pick(signif_idx_chx_kr_lp).get_data())
cluster_kr_gamma = ward_clustering_on_epochs(ep_kr_stand_gamma.copy().pick(signif_idx_chx_kr_gamma).get_data()-ep_kr_dev_gamma.copy().pick(signif_idx_chx_kr_gamma).get_data())
for clx_idx, clx in enumerate(sorted(set(cluster_kr_lp))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_kr_lp) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[2].scatter(x_lp[chx_in_cluster_true], y_lp[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} < 30 Hz', s=20)
for clx_idx, clx in enumerate(sorted(set(cluster_kr_gamma))):
    chx_in_cluster_indices = [i for i, c in enumerate(cluster_kr_gamma) if c == clx]
    chx_in_cluster_true = np.array(chx_in_cluster_indices)
    ax[2].scatter(x_gamma[chx_in_cluster_true], y_gamma[chx_in_cluster_true], color=cluster_colors[clx_idx], label=f'Cluster {clx} > 30 Hz', s=80,facecolors='none', edgecolors=cluster_colors [clx_idx],alpha=1)

fig.tight_layout()
plt.savefig("significant_clusters_comparison.png", dpi=300, bbox_inches='tight')
plt.show()